### 1 - Load the chunks and build the chunk index

In [6]:
import json
from minsearch import Index


def load_json(path):
    """Load a JSON file produced by an earlier notebook."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


chunks = load_json("chunks.json")

chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)
len(chunks)

295

### 2 - Define the search tool (with a call counter)

The function has a **type hint** and a **docstring** - frameworks read them to build the tool schema automatically. We also increment a counter on each call, so we can answer Q6 (how many times the agent searched).

In [7]:
# Counter to measure how many times the agent calls the tool.
search_calls = {"count": 0}


def search(query: str) -> list[dict]:
    """Search the course materials for chunks relevant to the query.

    Args:
        query: the search query text.

    Returns:
        A list of matching chunks (each with `filename` and `content`).
    """
    search_calls["count"] += 1
    return chunk_index.search(query, num_results=5)

### 3 - Build the agent with the search tool

Wire the tool, the LLM client (`gpt-5.4-mini`) and the instructions into a toyaikit runner, following the ToyAIKit lesson.

In [8]:
from openai import OpenAI
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner

# Register our search function as a tool.
tools = Tools()
tools.add_tool(search)

# Instructions that nudge the agent to search several times.
developer_prompt = (
    "You're a course teaching assistant. Answer the student's question "
    "using the search tool. Make multiple searches with different keywords "
    "before answering."
)

chat_interface = IPythonChatInterface()
openai_client = OpenAIClient(model="gpt-5.4-mini", client=OpenAI())

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    llm_client=openai_client,
)

### 4 - Check how run() expects its input



In [9]:
import inspect

print(inspect.signature(OpenAIResponsesRunner.run))

(self, previous_messages: list = None, stop_criteria: Callable = None) -> toyaikit.chat.runners.LoopResult


### 5 - Run the agent (interactive)

In [10]:
# Reset the counter right before the run so the count is clean.
search_calls["count"] = 0

# Interactive run: paste the question into the input box that appears.
runner.run()

-> Response received


-> Response received


KeyboardInterrupt: Interrupted by user

### 6 - Answer to Q6: how many times did the agent call `search`?

After the agent has answered and you ended the chat, read the counter.

Note: the agent decides this itself, so it varies between runs - pick the closest option (0 / 4 / 10 / 20).

In [11]:
print(f"The agent called search {search_calls['count']} times.")

The agent called search 4 times.
